In [ ]:
from atlite.gis import ExclusionContainer, shape_availability, padded_transform_and_shape
import time
import pandas as pd
import geopandas as gpd
from workflow.scripts.extra_functions import get_bounding

import rasterio as rio
from rasterio.enums import Resampling
from rasterio.features import shapes, rasterize
from rasterio.features import geometry_mask

import numpy as np
from numpy import empty
from scipy.ndimage import binary_dilation as dilation




In [ ]:
country_codes = {
    "Austria" : "AT",
    "Belgium" : "BE",
    "Bulgaria" : "BG", #
    "Switzerland" : "CH", #
    "Czech Republic" : "CZ", #
    "Germany" : "DE",
    "Denmark" : "DK",
    "Estonia" : "EE" ,
    "Spain" : "ES",
    "Finland" : "FI", #
    "France" : "FR",
    "United Kingdom" : "UK",
    "Greece" : "GR",
    "Croatia" : "HR", #
    "Hungary" : "HU", #
    "Ireland" : "IE",
    "Italy" : "IT",
    "Lithuania" : "LT",
    "Luxembourg" : "LU",
    "Latvia" : "LV",
    "Netherlands" : "NL",
    "Norway" : "NO",
    "Poland" : "PL", #
    "Portugal" : "PT",
    "Romania" : "RO", #
    "Sweden" : "SE",
    "Slovenia" : "SI", #
    "Slovakia" : "SK" #
}


In [ ]:
## method
method = snakemake.params.method # atlite

## input data
corine = snakemake.input.corine
scenario = snakemake.wildcards.scenario
europe_onshore_shp = snakemake.input.europe_onshore_shapefile

# params
corine_codes = snakemake.params.corine_codes
polygon_borders = snakemake.params.polygon_borders
input_scenarios = snakemake.params.input_scenarios["Settlement"]
target_crs = snakemake.params.target_crs # target EPSG

## output data
output_path = snakemake.output.settlement_path


In [ ]:
# resolution
res = 100

# borders
[rectx1, rectx2, recty1, recty2] = polygon_borders
polygon_bounds = get_bounding(target_crs,rectx1,rectx2,recty1,recty2)

# buffer
buffer = input_scenarios[scenario]

In [ ]:
europe = (
    gpd
    .read_file(europe_onshore_shp)
    .set_index(["index"])
    .loc[:,['geometry']]
    .to_crs(target_crs)
)

onshore = (
    europe
    .assign(area = 1)
    .dissolve(by='area')
    .reset_index()
)
onshore["area"] = onshore["area"].astype("uint8")

In [ ]:
t=time.time()

if method=="manual":
    # Create the whole area
    dst_transform, dst_shape = padded_transform_and_shape(
        bounds=rio.features.bounds(polygon_bounds), # cutout bounds
        res=res # resolution
    )

    # open corine
    src = rio.open(corine)
    raster_corine = src.read(1)
    src_transform = src.transform
    
    # reproject
    raster_reprojected = empty(dst_shape)
    rio.warp.reproject(
        raster_corine,
        raster_reprojected,
        src_transform=src_transform,
        src_crs=src.crs,
        dst_transform=dst_transform,
        dst_crs=target_crs,
        resampling=Resampling.nearest,
        num_threads=2
    )
    
    # only select some codes in corine
    raster_reprojected = np.where(np.isin(raster_reprojected,corine_codes), 1, 0)

    if isinstance(buffer, (int, float)): # it is only a value
        ## add buffers to every code
        iterations = int(buffer / res) + 1
        raster_buffered = dilation(raster_reprojected, iterations=iterations)

        europe_mask = ~geometry_mask(
            onshore.geometry,
            out_shape=dst_shape,
            transform=dst_transform,
            all_touched=True,
            )
        
        # select pixels only over onshore (inside Europe) area
        raster_buffered = europe_mask & raster_buffered

    else: # it is a file
        df = pd.read_csv(buffer)
        df["country"] = df["country"].replace(country_codes)
        df = df.set_index("country")
        buffers = df[[scenario]]

        ### Country specific values
        # Create a boolean mask containing settlements from CORINE
        # True: code is in the pixel
        # False: code is not in the pixel
        settlement_mask = raster_reprojected.astype(bool)

        # Create an array with zeros to then add excluded areas
        raster_buffered = np.zeros(dst_shape, dtype=np.uint8)

        # Create a loop through each country to buffer settlements
        for country, buffer in buffers.iterrows():
            country_name = country
            country_geom = europe.query("index==@country").geometry # get geometry of the country

            country_mask = ~geometry_mask(
                country_geom,
                out_shape=dst_shape,
                transform=dst_transform,
                all_touched=True,
                )

            # select pixels within a country and correspond to settlements
            country_settlements_mask = settlement_mask & country_mask
            
            # Add country specific buffer
            buffer_distance = int(buffer.values[0] / res) + 1
            buffered_settlements = dilation(country_settlements_mask, iterations=buffer_distance)
            
            # Add the excluded areas to the array
            raster_buffered[buffered_settlements] = 1

    
    raster_buffered = raster_buffered.astype("uint8")
    
    meta = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'nodata': None,
        'width': raster_buffered.shape[1],
        'height': raster_buffered.shape[0],
        'count': 1,
        'crs': target_crs,
        'transform': dst_transform,
        'compress': 'lzw'
    }
    
    # Save to file
    with rio.open(output_path, 'w', **meta) as dst:
        dst.write(raster_buffered, 1)
    print(time.time()-t)

200
72.81109261512756


In [ ]:

t=time.time()

if method=="atlite":

    exclusion_container = ExclusionContainer(crs=target_crs)
    exclusion_container.add_geometry(onshore.geometry)
    masked, _ = shape_availability(polygon_bounds.geometry, exclusion_container)
    raster_onshore = (~masked)


    
    if isinstance(buffer, (int, float)): # it is only a value
        exclusion_container = ExclusionContainer(crs=target_crs)
        # add buffer to every selected code
        exclusion_container.add_raster(corine, codes=corine_codes, buffer=buffer,crs=target_crs)
        # exclusion_container.add_geometry(onshore.geometry, invert=True)
        masked, transform = shape_availability(polygon_bounds.geometry, exclusion_container)
        raster_buffered = (~masked)

        raster_buffered = (raster_buffered & raster_onshore).astype("uint8")

    else:
        df = pd.read_csv(buffer)
        df["country"] = df["country"].replace(country_codes)
        df = df.set_index("country")
        buffers = df[[scenario]]

        exclusion_container = ExclusionContainer(crs=target_crs)
        # reproject and filter CORINE codes
        exclusion_container.add_raster(corine, codes=corine_codes,crs=target_crs)
        # exclusion_container.add_geometry(onshore.geometry, invert=True)
        masked, transform = shape_availability(polygon_bounds.geometry, exclusion_container)
        raster_buffered = (~masked)

        raster_buffered = (raster_buffered & raster_onshore)

        # Create a boolean mask containing excluded settlements
        # True: excluded codes
        # False: codes not excluded
        settlement_mask = (raster_buffered)

        # Create an array with zeros to then add excluded areas
        raster_buffered = empty(settlement_mask.shape, dtype=np.uint8)

        # Create a loop through each country to buffer settlements
        for country, buffer in buffers.iterrows():
            country_name = country
            country_geom = europe.query("index==@country").geometry # get geometry of the country

            country_mask = ~geometry_mask(
                country_geom,
                out_shape=dst_shape,
                transform=dst_transform,
                all_touched=True,
                )

            # select pixels within a country and correspond to settlements
            country_settlements_mask = settlement_mask & country_mask
            
            # Add country specific buffer
            buffer_distance = int(buffer.values[0] / res) + 1
            buffered_settlements = dilation(country_settlements_mask, iterations=buffer_distance)

            # Add the excluded areas to the array
            raster_buffered = raster_buffered | buffered_settlements
            

        raster_buffered = (raster_buffered).astype("uint8")

    meta = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'nodata': None,
        'width': raster_buffered.shape[1],
        'height': raster_buffered.shape[0],
        'count': 1,
        'crs': target_crs,
        'transform': transform,
        'compress': 'lzw'
    }
    
    # Save to file
    with rio.open(output_path, 'w', **meta) as dst:
        dst.write(raster_buffered, 1)
    print(time.time()-t)

200
588.7695279121399


In [ ]:
if method=="test":
    f = open(output_path, "x")